# Feature Engineering — Fraud_Data

**Objective.** Turn the cleaned transactions into a model-ready matrix: geolocation, time & velocity features, scaling, one-hot encoding, a leakage-safe train/test split, and SMOTE resampling on the training set only.

Output: `data/processed/fraud_features.csv`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable so `from src import ...` resolves.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

from src import config
config.ensure_dirs()
FIG = config.FIGURES_DIR


In [ ]:
from src import data_loader as dl, cleaning, feature_engineering as fe, geolocation as geo
df = cleaning.clean_fraud_data(dl.load_fraud_data())
print('Cleaned:', df.shape)

## 1. Geolocation integration

Convert IPs to integers and map each to a country via **range-based lookup** (`np.searchsorted`, O(n log m)). Requires `data/raw/IpAddress_to_Country.csv`.

In [ ]:
if config.IP_COUNTRY_RAW.exists():
    ip_country = dl.load_ip_country()
    df = geo.add_geolocation(df, ip_country)
    matched = (df['country'] != geo.UNKNOWN_COUNTRY).mean()
    print(f'IP->country match rate: {matched:.1%}')
    display(df[['ip_address', 'ip_int', 'country']].head())
else:
    print('IpAddress_to_Country.csv not found — adding ip_int only.')
    df = geo.add_ip_integer(df)
    df['country'] = geo.UNKNOWN_COUNTRY

In [ ]:
# Fraud patterns by country (top by volume).
if (df['country'] != geo.UNKNOWN_COUNTRY).any():
    rates = geo.fraud_rate_by_country(df, min_count=100)
    print('Highest-fraud-rate countries (>=100 txns):')
    display(rates.head(10))
    top = rates.head(15)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(y=top.index, x=top['fraud_rate'], ax=ax)
    ax.set(title='Fraud rate by country (top 15, >=100 txns)', xlabel='fraud rate')
    plt.savefig(FIG / 'fraud_rate_by_country.png', dpi=120, bbox_inches='tight')
    plt.show()

## 2. Time features

`hour_of_day`, `day_of_week`, and `time_since_signup` (hours between signup and purchase — near-zero gaps flag automated fraud).

In [ ]:
df = fe.add_time_features(df)
display(df[['signup_time','purchase_time','hour_of_day','day_of_week','time_since_signup']].head())
# time_since_signup differs sharply by class:
print(df.groupby('class')['time_since_signup'].median().rename('median_hours'))

## 3. Transaction frequency & velocity

Per-user and per-device counts, distinct users per device, and a 24-hour rolling purchase velocity per device.

In [ ]:
df = fe.add_frequency_features(df)
df = fe.add_velocity_features(df, window_hours=24.0)
vel_cols = ['user_transaction_count','device_transaction_count',
            'device_user_count','device_velocity_24h']
display(df[vel_cols].describe().round(2))
print('\nMean of each velocity feature by class:')
display(df.groupby('class')[vel_cols].mean().round(2))

## 4. Train/test split (before scaling & resampling)

We split **first** so that scaler fitting and SMOTE see only training data — preventing leakage. Stratified to preserve the fraud rate.

In [ ]:
from sklearn.model_selection import train_test_split
from src import transform, resampling

numeric_cols = ['purchase_value','age','hour_of_day','day_of_week',
                'time_since_signup','user_transaction_count',
                'device_transaction_count','device_user_count',
                'device_velocity_24h']
categorical_cols = ['source','browser','sex']
feature_cols = numeric_cols + categorical_cols

X = df[feature_cols].copy()
for c in categorical_cols:
    X[c] = X[c].astype(str)
y = df['class']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=config.RANDOM_STATE)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 5. Scaling + one-hot encoding

`StandardScaler` for numerics, `OneHotEncoder` for categoricals, bundled in a `ColumnTransformer` **fit on the training set only**.

In [ ]:
pre = transform.build_preprocessor(numeric_cols, categorical_cols, scaler='standard')
X_train_t = transform.fit_transform_frame(pre, X_train)
X_test_t = transform.transform_frame(pre, X_test)
print('Transformed train matrix:', X_train_t.shape)
X_train_t.head()

## 6. Handle class imbalance — SMOTE (training set only)

**Why SMOTE over undersampling?** With ~9% fraud, undersampling the majority would discard the bulk of legitimate transactions and the signal they carry. Random oversampling merely duplicates minority rows (overfitting risk). SMOTE synthesises new minority points by interpolating between near neighbours, enriching the minority region without literal duplicates. **Applied to the training fold only** so the test set keeps the real-world distribution.

In [ ]:
X_train_res, y_train_res = resampling.resample(X_train_t, y_train, method='smote')
report = resampling.resample_report(y_train, y_train_res)
print('Class distribution before vs after SMOTE (TRAIN ONLY):')
display(report)
print('Test set distribution is left UNCHANGED:')
display(resampling.class_distribution(y_test))

## 7. Persist processed features

In [ ]:
# Save the engineered (pre-scaling) frame for reuse + the split sizes.
out_cols = feature_cols + ['country', 'class']
df[out_cols].to_csv(config.FRAUD_FEATURES, index=False)
print('Saved', config.FRAUD_FEATURES)
print('Engineered feature columns:', feature_cols)

## 8. Summary

- **Geolocation:** IPs mapped to countries via range lookup; fraud rate varies by country.
- **Engineered features:** `hour_of_day`, `day_of_week`, `time_since_signup`, per-user/device counts, `device_user_count`, `device_velocity_24h`.
- **Transformation:** standardised numerics + one-hot categoricals via a leakage-safe `ColumnTransformer`.
- **Imbalance:** SMOTE balanced the **training** set (~90/10 → 50/50); the **test** set was untouched for honest evaluation.